# Make df for ml 

Basically decide what to do with missing values and categorical



In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
from pyprojroot import here

processed_data = here('data/processed_data')


In [3]:
contracts = pd.read_feather(processed_data / 'contracts_allfeatures.feather')

# categorical to ordinal and their missing values

In [4]:
#object columns
object_cols = contracts.select_dtypes(include=['object']).columns.tolist()
print(object_cols)

['file_code', 'contract_code', 'government_level', 'legal_framework', 'legal_fundament_simplified', 'procedure_venue', 'supplier_size', 'supply_type', 'supplier_name_clean', 'purchasing_unit_id', 'compliant_submission_period']


## legal framework

In [5]:
contracts['legal_framework'].value_counts()

national         1820721
international     400475
other              19852
Name: legal_framework, dtype: int64

In [6]:
legal_framework_dict = {
    'international': 2,
    'national': 1,
    'other': 0
}

contracts['legal_framework'] = contracts['legal_framework'].map(legal_framework_dict)
contracts['legal_framework'] = contracts['legal_framework'].fillna(0)

## legal fundament simplified

In [7]:
contracts['legal_fundament_simplified'].value_counts(dropna=False)

NaN      1488632
art41     507218
art42     281788
art43      31270
Name: legal_fundament_simplified, dtype: int64

In [8]:
contracts['legal_fundament_simplified'] = contracts['legal_fundament_simplified'].fillna('missing')

## government level

In [9]:
contracts['government_level'].value_counts(dropna=False)

federal_authority    2089420
state_authority       154940
local_authority        64548
Name: government_level, dtype: int64

In [10]:
government_level_dict = {
    'federal_authority': 2,
    'state_authority': 1,
    'local_authority': 0
}

contracts['government_level'] = contracts['government_level'].map(government_level_dict)

## procedure_venue

In [11]:
contracts['procedure_venue'].value_counts(dropna=False)

in-person     1020728
electronic     771676
mixed          448540
NaN             67964
Name: procedure_venue, dtype: int64

In [12]:
contracts['procedure_venue'] = contracts['procedure_venue'].fillna('missing')

## supplier_size

In [13]:
contracts['supplier_size'].value_counts(dropna=False)

NaN           669979
micro         524840
small         489807
medium        336137
not_mipyme    288145
Name: supplier_size, dtype: int64

In [14]:
contracts['supplier_size'] = contracts['supplier_size'].fillna('missing')

In [15]:
supplier_size_dict = {
    'not_mipyme': 3,
    'medium': 2,
    'small': 1,
    'micro': 0,
    'missing': -1
}
contracts['supplier_size'] = contracts['supplier_size'].map(supplier_size_dict)

## supply type

In [16]:
contracts['supply_type'].value_counts(dropna=False)

goods       1301679
services     791044
works        208684
NaN            7501
Name: supply_type, dtype: int64

In [17]:
contracts['supply_type'] = contracts['supply_type'].fillna('missing')

## compliant_submission_period

In [18]:
contracts['compliant_submission_period'].value_counts(dropna=False)

noncompliant    1009607
NaN              976887
compliant        225273
marginal          97141
Name: compliant_submission_period, dtype: int64

In [19]:
contracts['compliant_submission_period'] = contracts['compliant_submission_period'].fillna('missing')

In [20]:
compliant_submission_period_dict = {
    'compliant': 2,
    'marginal' : 1,
    'noncompliant': 0,
    'missing': -1
}
contracts['compliant_submission_period'] = contracts['compliant_submission_period'].map(compliant_submission_period_dict)

# ad hoc solutions for missing values or extra variables

In [21]:
contracts['contract_period'].isnull().sum()

326

In [22]:
#there are only 326 missing values in this variable, so we can just fill them with the median
contracts['contract_period'] = contracts['contract_period'].fillna(np.nanmedian(contracts['contract_period']))

#  nominal -> dummies

In [23]:
print(contracts['rf_single_bidder'].value_counts(dropna=False))
contracts['rf_single_bidder'] = contracts['rf_single_bidder'].fillna(-1)

NaN    2030204
0.0     248799
1.0      29905
Name: rf_single_bidder, dtype: int64


In [24]:
var2onehotencoding = ['legal_fundament_simplified', 'procedure_venue', 'supply_type']
contracts = pd.get_dummies(contracts, columns=var2onehotencoding, dtype=int)

# replacing missing values with arbitrary numbers

### tender period

In [25]:
contracts['tender_period'].describe()

count    1.907902e+06
mean    -1.536984e+01
std      7.689094e+01
min     -3.800000e+02
25%     -2.100000e+01
50%      3.000000e+00
75%      1.700000e+01
max      4.553960e+02
Name: tender_period, dtype: float64

In [26]:
contracts['tender_period'].min()

-380.0

In [27]:
contracts['tender_period'] = contracts['tender_period'].fillna(-400)

### rf_procedure_type

In [28]:
contracts['rf_procedure_type'].describe()

count    2.286176e+06
mean     8.058863e-01
std      3.622773e-01
min      0.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: rf_procedure_type, dtype: float64

In [29]:
contracts['rf_procedure_type'] = contracts['rf_procedure_type'].fillna(-1)

# neighbordhood 

In [30]:
contracts['neighborhood_propdirect'] = contracts['neighborhood_propdirect'].fillna(-1)

In [31]:
contracts['neighbourhood_cri'] = contracts['neighbourhood_cri'].fillna(-1)

## mad

In [32]:
contracts['mad'].describe()

count    2.045812e+06
mean     2.469531e-02
std      2.038970e-02
min      2.285825e-03
25%      1.061233e-02
50%      1.885878e-02
75%      3.207512e-02
max      1.796186e-01
Name: mad, dtype: float64

In [33]:
contracts['mad'] = contracts['mad'].fillna(-1)

# Check before saving

- No missing values (except in sanctioned variables)
- All numerical

In [34]:
[col for col in contracts.columns if contracts[col].isnull().any()]


['sanctionedA_C_all',
 'sanctionedA_C_max1',
 'sanctionedA_C_max2',
 'sanctionedA_C_max3',
 'sanctionedA_I_all',
 'sanctionedA_I_max1',
 'sanctionedA_I_max2',
 'sanctionedA_I_max3',
 'sanctionedB_C_all',
 'sanctionedB_C_max1',
 'sanctionedB_C_max2',
 'sanctionedB_C_max3',
 'sanctionedB_I_max1',
 'sanctionedB_I_max2',
 'sanctionedB_I_max3',
 'sanctionedE_EPN',
 'sanctionedE_AMLO']

In [35]:
contracts.select_dtypes(include=['object', 'category']).columns.tolist()

['file_code', 'contract_code', 'supplier_name_clean', 'purchasing_unit_id']

# Save contracts

There's one file with identifiers and other without

In [36]:
identifiers = [
            'file_code',
            'contract_code',
            'supplier_name_clean',
            'purchasing_unit_id',
            'compliant_submission_period',
            'contract_year'
                ]

In [37]:
#total contracts
contracts.to_feather(processed_data / 'contracts2ml.feather')